# RER Analysis and Visualization

This notebook demonstrates analysis and visualization of Reconstruction Error Ratios (RER)
following the framework from Marks et al. (2024).

## Topics Covered

1. RER interpretation and visualization
2. Dataset difficulty estimation (chi)
3. Noise rate estimation
4. Mislabel detection

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from model.ClasswiseAutoencoderManager import ClasswiseAutoencoderManager
from util.rer_metrics import (
    compute_rer,
    compute_dataset_chi,
    compute_class_chi,
    compute_chi_0,
    compute_chi_rand,
    estimate_noise_rate,
    compute_mislabel_threshold,
    detect_mislabels,
    compute_rer_statistics,
)

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Understanding RER

The Reconstruction Error Ratio (RER) measures how well a sample fits its assigned class:

$$\chi(x^c) = \frac{\Delta^c(x^c)}{\min_{c' \neq c} \Delta^{c'}(x^c)}$$

Where:
- $\Delta^c(x)$ is the reconstruction error from class c's autoencoder
- $x^c$ is a sample with label c

**Interpretation:**
- RER < 1: Sample fits its own class better than any other class (correctly classified)
- RER > 1: Sample fits another class better (potentially mislabeled or confusing)
- RER = 1: Sample is on the decision boundary

In [ ]:
# Create synthetic data with known properties
torch.manual_seed(42)
num_samples = 1000
num_classes = 5
feature_dim = 384

# Create class-specific features
class_centers = torch.randn(num_classes, feature_dim)
features = []
labels = []

for c in range(num_classes):
    n = num_samples // num_classes
    class_features = class_centers[c] + torch.randn(n, feature_dim) * 0.3
    features.append(class_features)
    labels.append(torch.full((n,), c, dtype=torch.long))

features = torch.cat(features)
labels = torch.cat(labels)

# Normalize
features = (features - features.min()) / (features.max() - features.min())

print(f"Data shape: {features.shape}")

In [ ]:
# Train autoencoders
manager = ClasswiseAutoencoderManager(
    num_classes=num_classes,
    feature_dim=feature_dim,
    n_components=10,
    hidden_dims=[256],
)

optimizer = torch.optim.Adam(manager.parameters(), lr=0.001)
for epoch in range(30):
    optimizer.zero_grad()
    loss, _ = manager.compute_total_loss(features, labels)
    loss.backward()
    optimizer.step()

print(f"Training complete. Final loss: {loss.item():.4f}")

In [ ]:
# Compute RER
manager.eval()
with torch.no_grad():
    all_errors = manager.compute_all_reconstruction_errors(features)
    rer_values = compute_rer(all_errors, labels)

stats = compute_rer_statistics(rer_values, labels)
print(f"Chi (mean RER): {stats['chi']:.4f}")

## 2. RER Distribution Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Overall RER histogram
ax = axes[0, 0]
ax.hist(rer_values.numpy(), bins=50, edgecolor='black', alpha=0.7, density=True)
ax.axvline(x=1.0, color='red', linestyle='--', linewidth=2, label='Decision boundary (RER=1)')
ax.axvline(x=stats['chi'], color='green', linestyle='-', linewidth=2, label=f'Chi = {stats["chi"]:.3f}')
ax.set_xlabel('RER')
ax.set_ylabel('Density')
ax.set_title('RER Distribution')
ax.legend()

# 2. Per-class RER boxplot
ax = axes[0, 1]
class_rer_data = [rer_values[labels == c].numpy() for c in range(num_classes)]
bp = ax.boxplot(class_rer_data, labels=[f'Class {c}' for c in range(num_classes)])
ax.axhline(y=1.0, color='red', linestyle='--', label='RER=1')
ax.set_ylabel('RER')
ax.set_title('Per-Class RER Distribution')
ax.legend()

# 3. Per-class chi bar plot
ax = axes[1, 0]
class_chis = [stats.get(f'chi_class_{c}', 0) for c in range(num_classes)]
colors = ['green' if chi < 1 else 'orange' if chi < 1.5 else 'red' for chi in class_chis]
ax.bar(range(num_classes), class_chis, color=colors)
ax.axhline(y=1.0, color='red', linestyle='--', label='RER=1')
ax.set_xlabel('Class')
ax.set_ylabel('Chi (mean RER)')
ax.set_title('Class Difficulty (Chi)')
ax.legend()

# 4. Reconstruction error heatmap
ax = axes[1, 1]
mean_errors = torch.zeros(num_classes, num_classes)
for c in range(num_classes):
    mask = labels == c
    mean_errors[c] = all_errors[mask].mean(dim=0)
    
im = ax.imshow(mean_errors.numpy(), cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label='Mean Reconstruction Error')
ax.set_xlabel('Autoencoder Class')
ax.set_ylabel('Sample Class')
ax.set_title('Reconstruction Error Matrix')
ax.set_xticks(range(num_classes))
ax.set_yticks(range(num_classes))

plt.tight_layout()
plt.show()

## 3. Noise Rate Estimation

The paper provides methods to estimate label noise rate from RER metrics:

$$\eta \approx \frac{\chi_0 - \chi_{rand}}{1 - \chi_{rand}}$$

In [ ]:
# Compute noise estimation metrics
chi_0 = compute_chi_0(all_errors, labels)
chi_rand = compute_chi_rand(all_errors, labels)
eta_est = estimate_noise_rate(chi_0, chi_rand)

print(f"Chi_0 (own class ratio): {chi_0:.4f}")
print(f"Chi_rand (best class ratio): {chi_rand:.4f}")
print(f"Estimated noise rate: {eta_est:.2%}")

In [ ]:
# Simulate adding label noise and track estimated noise rate
noise_rates = np.linspace(0, 0.4, 9)
estimated_rates = []

for noise_rate in noise_rates:
    # Add noise to labels
    noisy_labels = labels.clone()
    num_flip = int(noise_rate * len(labels))
    flip_idx = torch.randperm(len(labels))[:num_flip]
    noisy_labels[flip_idx] = torch.randint(0, num_classes, (num_flip,))
    
    # Compute RER with noisy labels
    chi_0 = compute_chi_0(all_errors, noisy_labels)
    chi_rand = compute_chi_rand(all_errors, noisy_labels)
    eta = estimate_noise_rate(chi_0, chi_rand)
    estimated_rates.append(eta)

plt.figure(figsize=(8, 6))
plt.plot(noise_rates * 100, [r * 100 for r in estimated_rates], 'b-o', label='Estimated')
plt.plot([0, 40], [0, 40], 'r--', label='Perfect estimation')
plt.xlabel('True Noise Rate (%)')
plt.ylabel('Estimated Noise Rate (%)')
plt.title('Noise Rate Estimation Accuracy')
plt.legend()
plt.grid(True)
plt.show()

## 4. Mislabel Detection

Samples with RER > threshold can be flagged as potentially mislabeled.

In [ ]:
# Add some artificial mislabels
torch.manual_seed(123)
noisy_labels = labels.clone()
num_mislabels = 50
mislabel_idx = torch.randperm(len(labels))[:num_mislabels]
true_labels_backup = noisy_labels[mislabel_idx].clone()
noisy_labels[mislabel_idx] = (noisy_labels[mislabel_idx] + torch.randint(1, num_classes, (num_mislabels,))) % num_classes

# Compute RER with noisy labels
rer_noisy = compute_rer(all_errors, noisy_labels)
chi_0_noisy = compute_chi_0(all_errors, noisy_labels)
chi_rand_noisy = compute_chi_rand(all_errors, noisy_labels)
eta_est_noisy = estimate_noise_rate(chi_0_noisy, chi_rand_noisy)

print(f"True mislabel rate: {num_mislabels / len(labels):.2%}")
print(f"Estimated noise rate: {eta_est_noisy:.2%}")

In [ ]:
# Compute threshold and detect mislabels
threshold = compute_mislabel_threshold(chi_0_noisy, eta_est_noisy)
detected = detect_mislabels(rer_noisy, threshold=threshold)

# Create ground truth mask
is_mislabeled = torch.zeros(len(labels), dtype=torch.bool)
is_mislabeled[mislabel_idx] = True

# Compute detection metrics
true_positives = (detected & is_mislabeled).sum().item()
false_positives = (detected & ~is_mislabeled).sum().item()
false_negatives = (~detected & is_mislabeled).sum().item()
true_negatives = (~detected & ~is_mislabeled).sum().item()

precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0

print(f"\nMislabel Detection Results:")
print(f"  Threshold: {threshold:.4f}")
print(f"  Detected as mislabeled: {detected.sum().item()}")
print(f"  True mislabels: {is_mislabeled.sum().item()}")
print(f"  True positives: {true_positives}")
print(f"  Precision: {precision:.2%}")
print(f"  Recall: {recall:.2%}")

In [ ]:
# Visualize mislabel detection
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RER distribution by label status
ax = axes[0]
ax.hist(rer_noisy[~is_mislabeled].numpy(), bins=30, alpha=0.7, label='Correctly labeled', density=True)
ax.hist(rer_noisy[is_mislabeled].numpy(), bins=30, alpha=0.7, label='Mislabeled', density=True)
ax.axvline(x=threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold={threshold:.2f}')
ax.set_xlabel('RER')
ax.set_ylabel('Density')
ax.set_title('RER Distribution by Label Status')
ax.legend()

# ROC-style curve for different thresholds
ax = axes[1]
thresholds = np.linspace(0.5, 3.0, 50)
precisions = []
recalls = []

for t in thresholds:
    det = rer_noisy > t
    tp = (det & is_mislabeled).sum().item()
    fp = (det & ~is_mislabeled).sum().item()
    fn = (~det & is_mislabeled).sum().item()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    precisions.append(prec)
    recalls.append(rec)

ax.plot(recalls, precisions, 'b-', linewidth=2)
ax.scatter([recall], [precision], color='red', s=100, zorder=5, label=f'Auto threshold (t={threshold:.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve for Mislabel Detection')
ax.legend()
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

## 5. Comparing Different Datasets

Chi values can be used to compare dataset difficulty across different datasets.

In [ ]:
# Simulate chi values for different datasets (in practice, compute from actual data)
datasets = {
    'MNIST': 0.45,
    'Fashion-MNIST': 0.72,
    'KMNIST': 0.68,
    'EMNIST-Digits': 0.52,
    'EMNIST-Balanced': 0.85,
}

plt.figure(figsize=(10, 6))
colors = ['green' if v < 0.6 else 'orange' if v < 0.8 else 'red' for v in datasets.values()]
bars = plt.bar(datasets.keys(), datasets.values(), color=colors, edgecolor='black')
plt.axhline(y=1.0, color='red', linestyle='--', label='RER=1 boundary')
plt.ylabel('Chi (Dataset Difficulty)')
plt.title('Dataset Difficulty Comparison')
plt.xticks(rotation=45)
plt.legend()

# Add value labels
for bar, val in zip(bars, datasets.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{val:.2f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:

1. **RER Computation**: How to compute and interpret RER values
2. **Dataset Difficulty**: Using chi (mean RER) to estimate classification difficulty
3. **Noise Estimation**: Estimating label noise rate from RER metrics
4. **Mislabel Detection**: Identifying potentially mislabeled samples

For production use, integrate these analyses with the Lightning training pipeline
using the `ClasswiseAETask` module.